# Short-List Promising Models & Fine-Tune the System

In [24]:
import pickle
import numpy as np
import pandas as pd

from sklearn import set_config
set_config(transform_output="pandas")

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

In [25]:
# Copied from the Preparation notebook (needed to load preprocessor.pkl)
class EarlySpringFeatures(BaseEstimator, TransformerMixin):
    def __init__(self, add_deltas=True, add_pre_march_trend=True, combine_oni=True, drop_redundant=True, use_year=False,
                 delta_features=("soil_temp_mean", "farm_avg_temp_mean", "om_temp_mean"),
                 redundant_features=("soil_thawed_days_jan", "soil_thawed_days_feb", "farm_freezing_days_feb")):
        self.add_deltas = add_deltas
        self.add_pre_march_trend = add_pre_march_trend
        self.combine_oni = combine_oni
        self.drop_redundant = drop_redundant
        self.use_year = use_year
        self.delta_features = delta_features
        self.redundant_features = redundant_features

    def fit(self, X, y=None):
        self.feature_names_out_ = self._engineer(X).columns.to_numpy()
        return self

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_out_

    def transform(self, X):
        return self._engineer(X)

    def _engineer(self, X):
        data = X.drop(columns="early_spring", errors="ignore")
        data["phil_pred"] = data["phil_pred"].astype(int)

        if self.add_deltas:
            for name in self.delta_features:
                data[f"{name}_delta"] = data[f"{name}_feb"] - data[f"{name}_jan"]

        if self.add_pre_march_trend:
            for prefix in ["farm_avg_temp", "soil_mean_soil_temp", "om_temperature_2m_mean"]:
                d1 = f"{prefix}_d_minus_1"
                d7 = f"{prefix}_d_minus_7"
                if d1 in data.columns and d7 in data.columns:
                    data[f"{prefix}_7d_trend"] = data[d1] - data[d7]

        if self.combine_oni:
            data["oni_winter_avg"] = data[["oni_prev_nov", "oni_prev_dec"]].mean(axis=1)
            data = data.drop(columns=["oni_prev_nov", "oni_prev_dec"])

        if self.drop_redundant:
            data = data.drop(columns=list(self.redundant_features), errors="ignore")

        if self.use_year:
            data["year"] = data.index.to_numpy()

        return data

In [26]:
train_set = pd.read_parquet("train_set.parquet")
X = train_set.drop(columns="early_spring")
y = train_set["early_spring"].astype(int)

with open("preprocessor.pkl", "rb") as f:
    preprocessor = pickle.load(f)

X_prepared = preprocessor.fit_transform(X, y)
print(X.shape, "->", X_prepared.shape)
y.value_counts()

(57, 43) -> (57, 47)


early_spring
0    43
1    14
Name: count, dtype: int64

In [27]:
# Stratified so every fold has some early springs
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [28]:
# Baseline: Phil's predictions
phil = X["phil_pred"].astype(int)
accuracy  = accuracy_score(y, phil)
precision = precision_score(y, phil)
recall    = recall_score(y, phil)
f1        = f1_score(y, phil)
print(f"Phil - accuracy: {accuracy:.3f}  precision: {precision:.3f}  recall: {recall:.3f}  f1: {f1:.3f}")

Phil - accuracy: 0.719  precision: 0.400  recall: 0.286  f1: 0.333


In [29]:
# class_weight="balanced" makes the models predict early spring more often (higher recall)
lr = LogisticRegression(class_weight="balanced", max_iter=5000)
lr.fit(X_prepared, y)
preds = lr.predict(X_prepared)
cv_preds = cross_val_predict(lr, X_prepared, y, cv=cv)

print(f"Train - accuracy: {accuracy_score(y, preds):.3f}  precision: {precision_score(y, preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, preds):.3f}  f1: {f1_score(y, preds):.3f}")
print(f"CV    - accuracy: {accuracy_score(y, cv_preds):.3f}  precision: {precision_score(y, cv_preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, cv_preds):.3f}  f1: {f1_score(y, cv_preds):.3f}")
confusion_matrix(y, cv_preds)

Train - accuracy: 0.930  precision: 0.778  recall: 1.000  f1: 0.875
CV    - accuracy: 0.649  precision: 0.312  recall: 0.357  f1: 0.333


array([[32, 11],
       [ 9,  5]])

In [30]:
sgd = SGDClassifier(class_weight="balanced", random_state=42)
sgd.fit(X_prepared, y)
preds = sgd.predict(X_prepared)
cv_preds = cross_val_predict(sgd, X_prepared, y, cv=cv)

print(f"Train - accuracy: {accuracy_score(y, preds):.3f}  precision: {precision_score(y, preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, preds):.3f}  f1: {f1_score(y, preds):.3f}")
print(f"CV    - accuracy: {accuracy_score(y, cv_preds):.3f}  precision: {precision_score(y, cv_preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, cv_preds):.3f}  f1: {f1_score(y, cv_preds):.3f}")
confusion_matrix(y, cv_preds)

Train - accuracy: 0.825  precision: 0.600  recall: 0.857  f1: 0.706
CV    - accuracy: 0.596  precision: 0.286  recall: 0.429  f1: 0.343


array([[28, 15],
       [ 8,  6]])

In [31]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_prepared, y)
preds = knn.predict(X_prepared)
cv_preds = cross_val_predict(knn, X_prepared, y, cv=cv)

print(f"Train - accuracy: {accuracy_score(y, preds):.3f}  precision: {precision_score(y, preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, preds):.3f}  f1: {f1_score(y, preds):.3f}")
print(f"CV    - accuracy: {accuracy_score(y, cv_preds):.3f}  precision: {precision_score(y, cv_preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, cv_preds):.3f}  f1: {f1_score(y, cv_preds):.3f}")
confusion_matrix(y, cv_preds)

Train - accuracy: 0.772  precision: 0.667  recall: 0.143  f1: 0.235
CV    - accuracy: 0.702  precision: 0.200  recall: 0.071  f1: 0.105


array([[39,  4],
       [13,  1]])

In [32]:
nb = GaussianNB()
nb.fit(X_prepared, y)
preds = nb.predict(X_prepared)
cv_preds = cross_val_predict(nb, X_prepared, y, cv=cv)

print(f"Train - accuracy: {accuracy_score(y, preds):.3f}  precision: {precision_score(y, preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, preds):.3f}  f1: {f1_score(y, preds):.3f}")
print(f"CV    - accuracy: {accuracy_score(y, cv_preds):.3f}  precision: {precision_score(y, cv_preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, cv_preds):.3f}  f1: {f1_score(y, cv_preds):.3f}")
confusion_matrix(y, cv_preds)

Train - accuracy: 0.772  precision: 1.000  recall: 0.071  f1: 0.133
CV    - accuracy: 0.754  precision: 0.500  recall: 0.143  f1: 0.222


array([[41,  2],
       [12,  2]])

In [33]:
dt = DecisionTreeClassifier(class_weight="balanced", min_samples_leaf=3, random_state=42)
dt.fit(X_prepared, y)
preds = dt.predict(X_prepared)
cv_preds = cross_val_predict(dt, X_prepared, y, cv=cv)

print(f"Train - accuracy: {accuracy_score(y, preds):.3f}  precision: {precision_score(y, preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, preds):.3f}  f1: {f1_score(y, preds):.3f}")
print(f"CV    - accuracy: {accuracy_score(y, cv_preds):.3f}  precision: {precision_score(y, cv_preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, cv_preds):.3f}  f1: {f1_score(y, cv_preds):.3f}")
confusion_matrix(y, cv_preds)

Train - accuracy: 0.947  precision: 0.867  recall: 0.929  f1: 0.897
CV    - accuracy: 0.649  precision: 0.312  recall: 0.357  f1: 0.333


array([[32, 11],
       [ 9,  5]])

In [34]:
rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf.fit(X_prepared, y)
preds = rf.predict(X_prepared)
cv_preds = cross_val_predict(rf, X_prepared, y, cv=cv)

print(f"Train - accuracy: {accuracy_score(y, preds):.3f}  precision: {precision_score(y, preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, preds):.3f}  f1: {f1_score(y, preds):.3f}")
print(f"CV    - accuracy: {accuracy_score(y, cv_preds):.3f}  precision: {precision_score(y, cv_preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, cv_preds):.3f}  f1: {f1_score(y, cv_preds):.3f}")
confusion_matrix(y, cv_preds)

Train - accuracy: 1.000  precision: 1.000  recall: 1.000  f1: 1.000
CV    - accuracy: 0.667  precision: 0.308  recall: 0.286  f1: 0.296


array([[34,  9],
       [10,  4]])

In [35]:
gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_prepared, y)
preds = gb.predict(X_prepared)
cv_preds = cross_val_predict(gb, X_prepared, y, cv=cv)

print(f"Train - accuracy: {accuracy_score(y, preds):.3f}  precision: {precision_score(y, preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, preds):.3f}  f1: {f1_score(y, preds):.3f}")
print(f"CV    - accuracy: {accuracy_score(y, cv_preds):.3f}  precision: {precision_score(y, cv_preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, cv_preds):.3f}  f1: {f1_score(y, cv_preds):.3f}")
confusion_matrix(y, cv_preds)

Train - accuracy: 1.000  precision: 1.000  recall: 1.000  f1: 1.000
CV    - accuracy: 0.632  precision: 0.182  recall: 0.143  f1: 0.160


array([[34,  9],
       [12,  2]])

In [36]:
svm_linear = SVC(kernel="linear", class_weight="balanced", random_state=42)
svm_linear.fit(X_prepared, y)
preds = svm_linear.predict(X_prepared)
cv_preds = cross_val_predict(svm_linear, X_prepared, y, cv=cv)

print(f"Train - accuracy: {accuracy_score(y, preds):.3f}  precision: {precision_score(y, preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, preds):.3f}  f1: {f1_score(y, preds):.3f}")
print(f"CV    - accuracy: {accuracy_score(y, cv_preds):.3f}  precision: {precision_score(y, cv_preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, cv_preds):.3f}  f1: {f1_score(y, cv_preds):.3f}")
confusion_matrix(y, cv_preds)

Train - accuracy: 0.947  precision: 0.824  recall: 1.000  f1: 0.903
CV    - accuracy: 0.702  precision: 0.412  recall: 0.500  f1: 0.452


array([[33, 10],
       [ 7,  7]])

In [37]:
svm_rbf = SVC(kernel="rbf", class_weight="balanced", random_state=42)
svm_rbf.fit(X_prepared, y)
preds = svm_rbf.predict(X_prepared)
cv_preds = cross_val_predict(svm_rbf, X_prepared, y, cv=cv)

print(f"Train - accuracy: {accuracy_score(y, preds):.3f}  precision: {precision_score(y, preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, preds):.3f}  f1: {f1_score(y, preds):.3f}")
print(f"CV    - accuracy: {accuracy_score(y, cv_preds):.3f}  precision: {precision_score(y, cv_preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, cv_preds):.3f}  f1: {f1_score(y, cv_preds):.3f}")
confusion_matrix(y, cv_preds)

Train - accuracy: 0.912  precision: 0.737  recall: 1.000  f1: 0.848
CV    - accuracy: 0.649  precision: 0.312  recall: 0.357  f1: 0.333


array([[32, 11],
       [ 9,  5]])

In [38]:
mlp = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=2000, random_state=42)
mlp.fit(X_prepared, y)
preds = mlp.predict(X_prepared)
cv_preds = cross_val_predict(mlp, X_prepared, y, cv=cv)

print(f"Train - accuracy: {accuracy_score(y, preds):.3f}  precision: {precision_score(y, preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, preds):.3f}  f1: {f1_score(y, preds):.3f}")
print(f"CV    - accuracy: {accuracy_score(y, cv_preds):.3f}  precision: {precision_score(y, cv_preds, zero_division=0):.3f}  "
      f"recall: {recall_score(y, cv_preds):.3f}  f1: {f1_score(y, cv_preds):.3f}")
confusion_matrix(y, cv_preds)

Train - accuracy: 1.000  precision: 1.000  recall: 1.000  f1: 1.000
CV    - accuracy: 0.632  precision: 0.000  recall: 0.000  f1: 0.000


array([[36,  7],
       [14,  0]])

In [39]:
for feature, importance in sorted(zip(rf.feature_names_in_, rf.feature_importances_), key=lambda x: x[1], reverse=True):
    print(f"{feature}: {importance:.3f}")

farm_snowfall_sum_feb: 0.064
om_precip_sum_feb: 0.059
om_snowfall_sum_feb: 0.054
soil_temp_mean_delta: 0.039
soil_temp_min_jan: 0.035
farm_precip_sum_jan: 0.035
om_temp_mean_feb: 0.033
om_temp_mean_delta: 0.032
soil_moisture_mean_feb: 0.031
farm_snowdepth_mean_feb: 0.030
om_snowfall_sum_jan: 0.026
om_sunshine_hours_sum_feb: 0.026
om_sunshine_hours_sum_jan: 0.025
farm_precip_sum_feb: 0.025
soil_moisture_mean_jan: 0.024
farm_avg_temp_mean_delta: 0.023
farm_avg_temp_mean_feb: 0.023
om_temp_min_feb: 0.023
soil_temp_mean_jan: 0.022
farm_min_temp_min_feb: 0.022
soil_temp_mean_feb: 0.022
farm_snow_cover_days_feb: 0.021
farm_max_temp_max_feb: 0.021
om_temp_max_feb: 0.021
soil_temp_max_jan: 0.021
oni_winter_avg: 0.019
soil_temp_min_feb: 0.019
om_precip_sum_jan: 0.017
om_temp_max_jan: 0.017
om_temp_min_jan: 0.016
farm_snowfall_sum_jan: 0.016
farm_snowdepth_mean_jan: 0.016
soil_temp_max_feb: 0.015
farm_snow_cover_days_jan: 0.014
om_freezing_days_jan: 0.014
farm_avg_temp_mean_jan: 0.013
farm_freez

In [40]:
# Positive = early spring, negative = late spring
for feature, coef in sorted(zip(lr.feature_names_in_, lr.coef_[0]), key=lambda x: abs(x[1]), reverse=True):
    print(f"{feature}: {coef:.3f}")

om_freezing_days_jan: 0.985
farm_avg_temp_mean_delta: -0.756
soil_temp_max_jan: 0.747
om_snowfall_sum_jan: 0.708
phil_pred: 0.682
soil_temp_mean_feb: -0.651
farm_max_temp_max_feb: 0.644
farm_snowfall_sum_feb: -0.601
farm_avg_temp_mean_jan: 0.580
om_freezing_days_feb: 0.571
om_temp_min_feb: -0.550
soil_temp_min_feb: 0.520
soil_temp_min_jan: -0.515
om_snowfall_sum_feb: -0.509
om_precip_sum_feb: -0.508
om_temp_max_feb: 0.505
om_temp_mean_delta: 0.477
farm_min_temp_min_jan: 0.455
soil_moisture_mean_feb: 0.445
soil_temp_mean_jan: -0.422
om_temp_min_jan: -0.403
om_precip_sum_jan: 0.398
soil_moisture_mean_jan: 0.394
om_temp_mean_feb: 0.359
farm_precip_sum_jan: 0.358
farm_snowdepth_mean_feb: -0.345
farm_snow_cover_days_jan: 0.342
om_temp_max_jan: 0.320
farm_avg_temp_mean_feb: -0.314
missingindicator_oni_winter_avg: 0.267
om_temp_mean_jan: -0.193
farm_max_temp_max_jan: -0.177
soil_temp_mean_delta: -0.169
oni_winter_avg: -0.166
farm_snowdepth_mean_jan: 0.154
om_sunshine_hours_sum_feb: 0.130
farm

In [ ]:
import warnings
from scipy.stats import loguniform, randint, uniform
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import make_scorer

# Some random combos of hyperparameters don't converge - hide those warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)
from sklearn.model_selection import RepeatedStratifiedKFold, RandomizedSearchCV, cross_validate

In [ ]:
# The log step is a FunctionTransformer inside the skewed pipeline - grab it so "log or not" can be tuned
log_transformer = preprocessor.named_steps["columns"].named_transformers_["skewed"].named_steps["log"]

In [ ]:
# Repeated folds give a more stable score on such a small dataset
tune_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
SCORING = {
    "f1": "f1",
    "recall": "recall",
    "precision": make_scorer(precision_score, zero_division=0),
    "accuracy": "accuracy",
}
REFIT = "f1"  # the metric used to pick the best hyperparameters

# Preprocessing hyperparameters, shared by every search
preprocessing_params = {
    "preprocessor__features__add_deltas": [True, False],
    "preprocessor__features__combine_oni": [True, False],
    "preprocessor__features__drop_redundant": [True, False],
    "preprocessor__features__use_year": [True, False],
    "preprocessor__columns__numeric__impute__strategy": ["mean", "median"],
    "preprocessor__columns__numeric__impute__add_indicator": [True, False],
    "preprocessor__columns__skewed__impute__strategy": ["mean", "median"],
    "preprocessor__columns__skewed__impute__add_indicator": [True, False],
    "preprocessor__columns__skewed__log": [log_transformer, "passthrough"],
}

def make_search(model, model_params, n_iter=100):
    """Randomized search over preprocessing + model hyperparameters, picking the best REFIT score."""
    pipe = Pipeline([("preprocessor", clone(preprocessor)), ("model", model)])
    return RandomizedSearchCV(pipe, {**preprocessing_params, **model_params}, n_iter=n_iter,
                              cv=tune_cv, scoring=SCORING, refit=REFIT, n_jobs=-1, random_state=42)

def show_best(search):
    """Print the best params and the CV scores of the best model."""
    best = search.best_index_
    for param, value in search.best_params_.items():
        print(f"{param}: {value}")
    print()
    for metric in SCORING:
        print(f"CV {metric}: {search.cv_results_[f'mean_test_{metric}'][best]:.3f}")

In [ ]:
svm_search = make_search(SVC(random_state=42), {
    "model__kernel": ["linear", "rbf", "poly"],
    "model__C": loguniform(1e-3, 1e2),
    "model__gamma": ["scale", "auto"],
    "model__degree": [2, 3],
    "model__class_weight": ["balanced", None, {0: 1, 1: 5}],
})
svm_search.fit(X, y)
show_best(svm_search)

In [ ]:
sgd_search = make_search(SGDClassifier(max_iter=5000, random_state=42), {
    "model__loss": ["hinge", "log_loss", "modified_huber"],
    "model__alpha": loguniform(1e-5, 1e0),
    "model__penalty": ["l2", "l1", "elasticnet"],
    "model__l1_ratio": uniform(0, 1),
    "model__class_weight": ["balanced", None, {0: 1, 1: 5}],
})
sgd_search.fit(X, y)
show_best(sgd_search)

In [ ]:
lr_search = make_search(LogisticRegression(solver="saga", max_iter=10000, random_state=42), {
    "model__C": loguniform(1e-3, 1e2),
    "model__l1_ratio": uniform(0, 1),  # 0 = L2, 1 = L1, in between = elastic net
    "model__class_weight": ["balanced", None, {0: 1, 1: 5}],
})
lr_search.fit(X, y)
show_best(lr_search)

In [ ]:
# Ensemble 1: random forest
rf_search = make_search(RandomForestClassifier(random_state=42), {
    "model__n_estimators": randint(50, 500),
    "model__max_depth": [2, 3, 5, 8, None],
    "model__min_samples_leaf": randint(1, 10),
    "model__max_features": ["sqrt", "log2", 0.5, 1.0],
    "model__bootstrap": [True, False],
    "model__class_weight": ["balanced", "balanced_subsample", None],
}, n_iter=50)
rf_search.fit(X, y)
show_best(rf_search)

In [ ]:
# Ensemble 2: majority vote of the three tuned models (each keeps its own tuned preprocessing)
voting = VotingClassifier([
    ("svm", svm_search.best_estimator_),
    ("sgd", sgd_search.best_estimator_),
    ("lr", lr_search.best_estimator_),
], voting="hard")
voting_scores = cross_validate(voting, X, y, cv=tune_cv, scoring=SCORING, n_jobs=-1)
for metric in SCORING:
    print(f"CV {metric}: {voting_scores[f'test_{metric}'].mean():.3f}")

In [ ]:
# Compare everything on the same CV folds
comparison = pd.DataFrame({
    name: {metric: search.cv_results_[f"mean_test_{metric}"][search.best_index_] for metric in SCORING}
    for name, search in [("SVM", svm_search), ("SGD", sgd_search), ("Logistic Regression", lr_search),
                         ("Random Forest", rf_search)]
}).T
comparison.loc["Voting"] = {metric: voting_scores[f"test_{metric}"].mean() for metric in SCORING}
# Baseline: always predict early spring
always_early = cross_validate(DummyClassifier(strategy="constant", constant=1), X, y, cv=tune_cv, scoring=SCORING)
comparison.loc["Always early spring"] = {metric: always_early[f"test_{metric}"].mean() for metric in SCORING}
comparison.sort_values(REFIT, ascending=False).round(3)

In [ ]:
# Save the final model (preprocessor is the first step of the pipeline, so it's included)
# Change this to whichever model you decide is best
final_model = svm_search.best_estimator_
final_model.fit(X, y)

with open("model.pkl", "wb") as f:
    pickle.dump(final_model, f)